In [1]:
from google.colab import files
uploaded = files.upload()

Saving arkose_passages_4000_v2.csv to arkose_passages_4000_v2.csv


In [2]:
from google.colab import files
uploaded = files.upload()

Saving arkose_clients.csv to arkose_clients.csv


In [3]:
import pandas as pd

clients = pd.read_csv("arkose_clients.csv")
passages = pd.read_csv("arkose_passages_4000_v2.csv")

In [29]:
# vérifier les types
passages.info()
clients.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date Passage   4000 non-null   datetime64[ns]
 1   Etablissement  4000 non-null   object        
 2   ID Client      4000 non-null   int64         
 3   Type Forfait   4000 non-null   object        
 4   Designation    4000 non-null   object        
 5   Quantite       4000 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 187.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ID Client                  99 non-null     int64 
 1   Etablissement Inscription  99 non-null     object
 2   Date Inscription           99 non-null     object
 3   Date de naissance          99 non

In [50]:
# vérifier les nulls
passages.isnull().sum()



,0
Date Passage,0
Etablissement,0
ID Client,0
Type Forfait,0
Designation,0
Quantite,0


In [51]:
clients.isnull().sum()

,0
ID Client,0
Etablissement Inscription,0
Date Inscription,0
Date de naissance,0


In [48]:
# vérifier doublons
passages.duplicated().sum()

np.int64(0)

In [52]:
clients.duplicated().sum()

np.int64(0)

In [43]:
# convertir dates
passages["Date Passage"] = pd.to_datetime(passages["Date Passage"], errors="coerce")

In [4]:
import sqlite3

conn = sqlite3.connect("arkose.db")

clients.to_sql("clients", conn, index=False, if_exists="replace")
passages.to_sql("passages", conn, index=False, if_exists="replace")

4000

In [45]:
query = """
SELECT
    strftime('%Y', "Date Passage") AS annee,
    strftime('%m', "Date Passage") AS mois,
    SUM(Quantite) AS volume
FROM passages
GROUP BY annee, mois
ORDER BY annee, mois;
"""

df_volume_mois = pd.read_sql(query, conn)
df_volume_mois.head(10)

,annee,mois,volume
0,2020,01,99
1,2020,02,105
2,2020,03,118
3,2020,04,118
4,2020,05,104
5,2020,06,92
6,2020,07,112
7,2020,08,121
8,2020,09,106
9,2020,10,115


In [46]:
query2 = """
SELECT
    "Type Forfait",
    SUM(Quantite) AS volume
FROM passages
GROUP BY "Type Forfait"
ORDER BY volume DESC;
"""

df_volume_forfait = pd.read_sql(query2, conn)
df_volume_forfait

,Type Forfait,volume
0,Annuel,1345
1,Unité,1337
2,Mensuel,1318


In [21]:
query3 = """
SELECT
    Etablissement,
    SUM(Quantite) AS volume
FROM passages
GROUP BY Etablissement
ORDER BY volume DESC;
"""

df_volume_etab = pd.read_sql(query3, conn)
df_volume_etab

,Etablissement,volume
0,Arkose Toulouse,1029
1,Arkose Lyon,1017
2,Arkose Bordeaux,984
3,Arkose Paris,970


In [57]:
derniere_visite = passages.groupby("ID Client")["Date Passage"].max().reset_index()
derniere_visite.rename(columns={"Date Passage": "derniere_visite"}, inplace=True)
derniere_visite.head(10)

,ID Client,derniere_visite
0,1000147,2020-03-04 01:29:00
1,1000155,2020-11-17 10:49:00
2,1000618,2022-11-21 07:38:00
3,1000668,2022-11-11 06:39:00
4,1000712,2020-02-12 19:29:00
5,1000805,2021-02-06 00:31:00
6,1000865,2020-08-02 20:18:00
7,1001110,2021-03-14 09:19:00
8,1001395,2020-06-05 16:17:00
9,1001507,2021-05-31 21:58:00


In [60]:
today = pd.Timestamp.today().normalize()

derniere_visite["recence_jours"] = (today - derniere_visite["derniere_visite"]).dt.days
derniere_visite.head(10)

,ID Client,derniere_visite,recence_jours
0,1000147,2020-03-04 01:29:00,2237
1,1000155,2020-11-17 10:49:00,1979
2,1000618,2022-11-21 07:38:00,1245
3,1000668,2022-11-11 06:39:00,1255
4,1000712,2020-02-12 19:29:00,2258
5,1000805,2021-02-06 00:31:00,1898
6,1000865,2020-08-02 20:18:00,2086
7,1001110,2021-03-14 09:19:00,1862
8,1001395,2020-06-05 16:17:00,2144
9,1001507,2021-05-31 21:58:00,1784


In [69]:
# date de référence = dernière date du dataset
reference_date = passages["Date Passage"].max()

# récence
derniere_visite["recence_jours"] = (reference_date - derniere_visite["derniere_visite"]).dt.days

# segmentation
def segment_client(recence):
    if pd.isna(recence):
        return "Inconnu"
    elif recence <= 30:
        return "Très actif"
    elif recence <= 60:
        return "Actif"
    elif recence <= 90:
        return "À risque"
    else:
        return "Inactif"

derniere_visite["segment_client"] = derniere_visite["recence_jours"].apply(segment_client)

# résultats
print(reference_date)
print(derniere_visite["segment_client"])

2022-12-30 20:08:00
0       Inactif
1       Inactif
2         Actif
3         Actif
4       Inactif
         ...   
3989    Inactif
3990    Inactif
3991    Inactif
3992    Inactif
3993    Inactif
Name: segment_client, Length: 3994, dtype: object


In [67]:
print(derniere_visite["segment_client"].value_counts())

segment_client
Inactif       3641
À risque       125
Très actif     119
Actif          109
Name: count, dtype: int64


In [71]:
ordre_segments = ["Très actif", "Actif", "À risque", "Inactif"]

segment_counts = derniere_visite["segment_client"].value_counts().reindex(ordre_segments, fill_value=0).reset_index()
segment_counts.columns = ["segment_client", "nb_clients"]

segment_counts["pourcentage"] = (
    segment_counts["nb_clients"] / segment_counts["nb_clients"].sum() * 100
).round(2)

segment_counts


,segment_client,nb_clients,pourcentage
0,Très actif,119,2.98
1,Actif,109,2.73
2,À risque,125,3.13
3,Inactif,3641,91.16


In [72]:
segment_counts.style.hide(axis="index")

segment_client,nb_clients,pourcentage
Très actif,119,2.980000
Actif,109,2.730000
À risque,125,3.130000
Inactif,3641,91.160000
